# pa2-dpo-uf — Task 4: from-scratch DPO on UltraFeedback
Settings: GPU T4 x2, Internet on. **Add Input → Notebook Output → pa2-sft** (latest version).
Launch with **Save Version → Save & Run All (Commit)**. ~2–2.5 h. Policy on cuda:0, frozen reference on cuda:1.

In [ ]:
# ---- Setup: identical first cell in every notebook (a commit starts on a fresh machine) ----
GH_USER, REPO = "mohdddd", "small-model-alignment-starter"
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/{GH_USER}/{REPO}.git repo
%cd /kaggle/working/repo
!git log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
import os, sys, json, glob, shutil, subprocess
sys.path.insert(0, "/kaggle/working/repo/src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("outputs", exist_ok=True)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
from pa2.utils import find_one
SFT = os.path.dirname(find_one("sft/best/model.safetensors"))
print("SFT best checkpoint:", SFT)
print(open(os.path.join(SFT, "checkpoint_meta.json")).read())

In [ ]:
!python scripts/prepare_data.py --which uf --out_dir outputs/data
print(open('outputs/data/data_stats.json').read())

In [ ]:
!python scripts/train_dpo.py --sft_path {SFT} --train_file outputs/data/uf_train.jsonl --val_file outputs/data/uf_val.jsonl --out_dir outputs/dpo_uf

In [ ]:
from IPython.display import Image, display
def show(path):
    display(Image(path))
!python scripts/plot_metrics.py --run_dir outputs/dpo_uf --kind dpo --title "UltraFeedback DPO"
show('outputs/dpo_uf/dpo_curves.png')
print(open('outputs/dpo_uf/summary.json').read())

In [ ]:
!cd outputs && zip -qr /kaggle/working/dpo_uf_run_outputs.zip dpo_uf/metrics.jsonl dpo_uf/config.json dpo_uf/summary.json dpo_uf/dpo_curves.png dpo_uf/best/checkpoint_meta.json data/data_stats.json
!du -sh /kaggle/working/repo/outputs/* 2>/dev/null; df -h /kaggle/working | tail -1